# The Live Agentic Planner — Design Deep Dive
### Stage 3 of the Control Loop: How the Agent Reasons, What It Outputs, and How It's Kept Safe

**This notebook contains no executable code on purpose.** It is a conceptual and design walkthrough of the agent itself — the prompt it receives, the structured plan it must return, and the two-stage safety check that stands between its output and the real network.

The actual, runnable implementation lives in two parallel Python modules — open them alongside this notebook:

- **`src/agent_planner.py`** — builds the prompt, calls the LLM, runs the bounded retry loop.
- **`src/safety_layer.py`** — the deterministic validator and fallback policy.
- **`src/schemas.py`** — the exact data contract (`AllocationPlan`) both of the above share.

Every section below ends with a pointer to the specific function/class that implements it.


## 1. What "Agentic" Means Here

A single LLM API call that answers a question is not an agent. What makes this component agentic is that it:

- Operates **inside a control loop**, repeatedly, once per timestep — not once, on demand.
- Receives **live state** (from the Digital Twin and Forecaster) rather than a static prompt.
- Must produce a **structured decision**, not free-form prose — the output is consumed by code, not read by a human first.
- **Retries itself** with feedback when its own output is rejected, before anything falls back to a simpler policy.
- Is bounded by an external, non-negotiable **safety layer** that it cannot talk its way around — its autonomy is real but deliberately limited.

That combination — live state in, a structured action out, self-correction, and an external safety boundary — is the working definition of "agentic" used throughout this project.


## 2. Inputs to the Agent

At every control-loop timestep, the planner is given exactly four things:

1. **Current allocations** — the bandwidth currently assigned to each slice (URLLC, eMBB).
2. **Latest twin metrics** — per-slice observed latency and throughput for the timestep that just completed.
3. **The forecaster's prediction** — mean $\mu$ and standard deviation $\sigma$ of next-timestep traffic, per slice (from `02_digital_twin.ipynb` and `03_forecaster.ipynb` respectively).
4. **Hard network rules** — total capacity, the minimum bandwidth guarantee for URLLC, and the maximum allowed change per step (to prevent oscillation).

Nothing else is passed in. In particular, the agent does **not** see raw historical traffic directly — that summarisation is the forecaster's job, so the planner's prompt stays short, cheap, and focused on the decision itself.

*Implemented by:* `src/agent_planner.py :: build_state_context()`


## 3. The Prompt Contract

The system prompt frames the LLM explicitly as a network operator making one allocation decision, not as a general-purpose chatbot. Three design choices matter most:

- **Role framing.** The prompt states the operator role, the hard rules, and the consequence of an URLLC breach up front — this is what "reasoning like a human network operator" (from the README) actually means in practice: the same information a real NOC engineer would have on their screen.
- **Explicit output-only instruction.** The prompt forbids any prose outside the JSON object. This is a deliberate, testable constraint (Section 5) — not a suggestion.
- **One worked example.** A single few-shot example of a well-formed input/output pair is included, since structured-output reliability improves noticeably with one concrete example versus a bare schema description alone.

*Implemented by:* `src/agent_planner.py :: build_prompt()`


## 4. The Output Contract — Structured Allocation Plan

The agent must return one JSON object, matching a fixed schema, every time:

| Field | Type | Purpose |
|---|---|---|
| `timestep` | int | Which control cycle this plan is for — used for logging/replay. |
| `allocations` | object `{slice_name: Mbps}` | The actual decision — bandwidth per slice for the next timestep. |
| `reasoning` | string | Plain-English justification — this is what feeds the dashboard's "agent reasoning" panel from the roadmap. |
| `risk_flag` | string enum (`"low"`, `"medium"`, `"high"`) | The agent's own self-reported confidence, used as an extra (non-authoritative) signal — the safety layer never trusts this field on its own. |

This is deliberately the *only* thing that crosses the boundary from the LLM into the rest of the system. Nothing else the model says is read anywhere downstream.

*Implemented by:* `src/schemas.py :: AllocationPlan`


## 5. The Two-Stage Safety Check

Every returned plan is checked twice, in order, before it is trusted:

**Stage A — Syntactic check.** Is the response valid JSON? Does it match the `AllocationPlan` schema exactly — right field names, right types, no missing fields? An LLM that returns malformed JSON, extra prose, or a wrong field name fails here, immediately, before any network logic is even considered.

**Stage B — Semantic / constraint check.** Given a *syntactically* valid plan, does it actually make physical sense?
- Do the allocations sum to no more than total capacity?
- Is URLLC's minimum guaranteed bandwidth still respected?
- Are all allocations non-negative?
- Is the change from the previous timestep within the maximum-step-change bound (anti-oscillation)?

A plan can pass Stage A and still fail Stage B — e.g. a well-formed JSON object that happens to allocate more bandwidth than exists.

*Implemented by:* `src/safety_layer.py :: validate_schema()` (Stage A) and `validate_constraints()` (Stage B)


## 6. The Bounded Retry Loop — This Project's Add-On

The naive design is "check once, fall back on failure." This project does one better: on either check's failure, the **specific validation error** (e.g. `"allocations sum to 118 Mbps, capacity is 100 Mbps"`) is fed back into the LLM as additional context, and the model is re-prompted — bounded to a small, fixed number of retries (2, by default).

This matters because most LLM planning failures are not reasoning failures — they're small, mechanical formatting slips. A targeted retry with the exact error message fixes the large majority of these without ever touching the fallback policy, while the bound keeps worst-case control-loop latency predictable.

*Implemented by:* `src/agent_planner.py :: plan_with_retry()`


## 7. The Fallback Policy

If every retry is exhausted and the plan still fails either check, the system does **not** act on it. Instead it falls back to a deterministic, pre-defined safe policy for that one timestep only — e.g. the last known-good allocation, or a fixed proportional-fair split that always respects the URLLC minimum guarantee by construction.

This is the single most important guarantee in the whole pipeline: **no unverified LLM output can ever reach the actuation stage.** The fallback is boring and predictable on purpose.

*Implemented by:* `src/safety_layer.py :: fallback_policy()`


## 8. Explainability Hook — Where `reasoning` Actually Goes

The `reasoning` field from Section 4 is not decorative. It's logged alongside every timestep's decision and is exactly what the roadmap's live dashboard renders in its "agent reasoning" panel — the feature that turns this from a script into something a NOC engineer could actually watch and audit in real time, which is the whole justification for choosing an LLM over a black-box policy in the first place (Section 3 of `00_overview.ipynb`).


## 9. The Three Comparison Arms

The planner described above is only one of three systems run against the same twin and the same traffic, in `06_baseline_comparison.ipynb`:

1. **Static Baseline** — fixed split, never adapts. The control arm.
2. **Legacy PPO/RL Agent** — the original prototype's reinforcement-learning policy, kept and re-run rather than discarded, so its opacity/reactivity limitations (Section 3, `00_overview.ipynb`) are demonstrated empirically, not just argued.
3. **Live Agentic System** — everything described in this notebook.

*Implemented by:* `src/baseline_agents.py`


## 10. Design Trade-offs Worth Being Upfront About

- **Control-loop latency.** Every timestep now includes a real LLM API call in the critical path. This bounds how fine-grained the control loop's timestep can be in this simulated setting — a real deployment would need either a much faster/local model or a longer control interval, and this is a known, stated limitation rather than something the project pretends away.
- **Low temperature, not zero.** The planner is called at a low sampling temperature to make its allocations consistent across similar states, not fully deterministic — full determinism would remove any benefit of the LLM reasoning over the input each time.
- **Cost/rate limits.** Every simulated timestep is one API call; this is a real, practical constraint on how long an experiment run can be before it needs to be batched or the model swapped for a cheaper one.

*Implemented by:* configuration constants at the top of `src/agent_planner.py`


## 11. Where Everything Lives — Concept to Code Map

| Concept (this notebook) | File | Function / Class |
|---|---|---|
| State context assembly (Section 2) | `src/agent_planner.py` | `build_state_context()` |
| Prompt construction (Section 3) | `src/agent_planner.py` | `build_prompt()` |
| Output schema (Section 4) | `src/schemas.py` | `AllocationPlan` |
| Syntactic check (Section 5A) | `src/safety_layer.py` | `validate_schema()` |
| Semantic/constraint check (Section 5B) | `src/safety_layer.py` | `validate_constraints()` |
| Bounded retry loop (Section 6) | `src/agent_planner.py` | `plan_with_retry()` |
| Fallback policy (Section 7) | `src/safety_layer.py` | `fallback_policy()` |
| Comparison baselines (Section 9) | `src/baseline_agents.py` | `StaticBaselineAgent`, `LegacyPPOAgent` |

---
*Next: → `02_digital_twin.ipynb`*
